# PyDI Data Integration Workflow: Papers

This notebook consolidates the paper use case into one executable workflow: source-schema mapping, blocking, entity matching, and data fusion.

## Table of Contents

1. Schema matching and source-schema translation
2. Blocking
3. Entity matching
4. Data fusion

## Part 1: Schema Matching

In [1]:
from pathlib import Path
import json
import logging
import math
import re
import sys

import numpy as np
import pandas as pd
from IPython.display import display

# Resolve paths whether the notebook is run from the repo root or from usecases/papers.
BASE_DIR = Path.cwd()
if not (BASE_DIR / "input" / "data").exists():
    candidate = BASE_DIR / "usecases" / "papers"
    if candidate.exists():
        BASE_DIR = candidate.resolve()

REPO_ROOT = BASE_DIR.parents[1]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

DATA_DIR = BASE_DIR / "input" / "data"
OUTPUT_DIR = BASE_DIR / "output"
SCHEMA_OUTPUT_DIR = OUTPUT_DIR / "schema_matching"
EM_OUTPUT_DIR = OUTPUT_DIR / "entity_matching"
FUSION_OUTPUT_DIR = OUTPUT_DIR / "data_fusion"

for path in [OUTPUT_DIR, SCHEMA_OUTPUT_DIR, EM_OUTPUT_DIR, FUSION_OUTPUT_DIR, OUTPUT_DIR / "logs"]:
    path.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="[%(levelname)-5s] %(name)s - %(message)s",
    handlers=[logging.FileHandler(OUTPUT_DIR / "logs" / "papers_workflow.log"), logging.StreamHandler()],
    force=True,
)

print("Base directory:", BASE_DIR)
print("Input data:", DATA_DIR)

Base directory: /Users/aaronsteiner/Documents/GitHub/PyDI/usecases/papers
Input data: /Users/aaronsteiner/Documents/GitHub/PyDI/usecases/papers/input/data


In [2]:
# Source files expected by the original schema-mapping cells.
# Path adaptation only: the original notebook used open_alex_filtered.jsonl,
# dblp_filtered.jsonl, and crossref_filtered.jsonl in the SchemaMapping folder.
OPENALEX_FILTERED_PATH = DATA_DIR / "open_alex.jsonl"
DBLP_FILTERED_PATH = DATA_DIR / "dblp.jsonl"
CROSSREF_FILTERED_PATH = DATA_DIR / "crossref.jsonl"

for name, path in {
    "open_alex_filtered": OPENALEX_FILTERED_PATH,
    "dblp_filtered": DBLP_FILTERED_PATH,
    "crossref_filtered": CROSSREF_FILTERED_PATH,
}.items():
    print(f"{name}: {path}")
    if path.exists():
        preview = pd.read_json(path, lines=True, nrows=2)
        print(f"  columns: {list(preview.columns)}")

open_alex_filtered: /Users/aaronsteiner/Documents/GitHub/PyDI/usecases/papers/input/data/open_alex.jsonl
  columns: ['doi', 'type', 'title', 'authors', 'publication_year', 'journal', 'publisher', 'keywords', 'volume', 'issue', 'first_page', 'last_page', 'referenced_works_count', 'cited_by_count']
dblp_filtered: /Users/aaronsteiner/Documents/GitHub/PyDI/usecases/papers/input/data/dblp.jsonl
  columns: ['doi', 'type', 'title', 'authors', 'publication_year', 'journal', 'publisher', 'keywords', 'volume', 'issue', 'first_page', 'last_page', 'referenced_works_count', 'cited_by_count']
crossref_filtered: /Users/aaronsteiner/Documents/GitHub/PyDI/usecases/papers/input/data/crossref.jsonl
  columns: ['doi', 'type', 'title', 'authors', 'publication_year', 'journal', 'publisher', 'keywords', 'volume', 'issue', 'first_page', 'last_page', 'referenced_works_count', 'cited_by_count']


In [3]:
TARGET_SCHEMA_PATH = BASE_DIR / "input" / "schemamatching" / "target_schema.json"
with TARGET_SCHEMA_PATH.open("r", encoding="utf-8") as handle:
    target_schema = json.load(handle)

TARGET_SCHEMA = [
    column
    for column in target_schema["properties"].keys()
    if column != "id"
]

schema_correspondences = pd.DataFrame(
    [
        # OpenAlex
        ("open_alex", "doi", "target", "doi"),
        ("open_alex", "type", "target", "type"),
        ("open_alex", "title", "target", "title"),
        ("open_alex", "authorships", "target", "authors"),
        ("open_alex", "publication_year", "target", "publication_year"),
        ("open_alex", "primary_location", "target", "journal/publisher"),
        ("open_alex", "keywords", "target", "keywords"),
        ("open_alex", "biblio", "target", "volume/issue/first_page/last_page"),
        ("open_alex", "referenced_works_count", "target", "referenced_works_count"),
        ("open_alex", "cited_by_count", "target", "cited_by_count"),
        # DBLP
        ("dblp", "_doi_list", "target", "doi"),
        ("dblp", "_type", "target", "type"),
        ("dblp", "title", "target", "title"),
        ("dblp", "author", "target", "authors"),
        ("dblp", "year", "target", "publication_year"),
        ("dblp", "journal/booktitle", "target", "journal"),
        ("dblp", "publisher", "target", "publisher"),
        ("dblp", "volume", "target", "volume"),
        ("dblp", "number", "target", "issue"),
        ("dblp", "pages", "target", "first_page/last_page"),
        # Crossref
        ("crossref", "DOI", "target", "doi"),
        ("crossref", "type", "target", "type"),
        ("crossref", "title", "target", "title"),
        ("crossref", "author", "target", "authors"),
        ("crossref", "published", "target", "publication_year"),
        ("crossref", "container-title", "target", "journal"),
        ("crossref", "publisher", "target", "publisher"),
        ("crossref", "subject", "target", "keywords"),
        ("crossref", "volume", "target", "volume"),
        ("crossref", "issue", "target", "issue"),
        ("crossref", "page", "target", "first_page/last_page"),
        ("crossref", "reference-count", "target", "referenced_works_count"),
        ("crossref", "is-referenced-by-count", "target", "cited_by_count"),
    ],
    columns=["source_dataset", "source_column", "target_dataset", "target_column"],
)

print("Loaded target schema:", TARGET_SCHEMA_PATH)
print("Target fields:", ["id", *TARGET_SCHEMA])
display(schema_correspondences)

Loaded target schema: /Users/aaronsteiner/Documents/GitHub/PyDI/usecases/papers/input/schemamatching/target_schema.json
Target fields: ['id', 'doi', 'type', 'title', 'authors', 'publication_year', 'journal', 'publisher', 'keywords', 'volume', 'issue', 'first_page', 'last_page', 'referenced_works_count', 'cited_by_count']


,source_dataset,source_column,target_dataset,target_column
0,open_alex,doi,target,doi
1,open_alex,type,target,type
2,open_alex,title,target,title
3,open_alex,authorships,target,authors
4,open_alex,publication_year,target,publication_year
5,open_alex,primary_location,target,journal/publisher
6,open_alex,keywords,target,keywords
7,open_alex,biblio,target,volume/issue/first_page/last_page
8,open_alex,referenced_works_count,target,referenced_works_count
9,open_alex,cited_by_count,target,cited_by_count


In [4]:
def normalize_doi(value):
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return None
    value = str(value).strip().lower()
    for prefix in ("https://doi.org/", "http://doi.org/", "doi:"):
        if value.startswith(prefix):
            value = value[len(prefix):]
    return value.rstrip(".") or None


# Exact helper from finding_a_matcher.ipynb cell 4.
def normalize_authors_list(data):
    """
    Normalizes various author string/list formats into a clean list of names.
    Handles strings, lists, np.nan, AND np.ndarray.
    """
    if isinstance(data, (list, np.ndarray)):
        return [str(item).strip().lower() for item in data if str(item).strip()]
    if pd.isna(data):
        return []
    text = str(data).strip().lower().strip('[]')
    return [author.strip() for author in text.split(',') if author.strip()]

In [5]:
from collections.abc import Mapping, Sequence
from PyDI.io import load_json

TARGET_COLUMNS = ["doi", *[c for c in TARGET_SCHEMA if c != "doi"]]


def _already_target_shaped(df: pd.DataFrame) -> bool:
    return set(TARGET_COLUMNS).issubset(df.columns)


def _write_schema_output(df: pd.DataFrame, new_name: str):
    SCHEMA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    output_path = SCHEMA_OUTPUT_DIR / f"{new_name}.jsonl"
    out = df.copy()
    for col in TARGET_COLUMNS:
        if col not in out.columns:
            out[col] = np.nan
    out = out[TARGET_COLUMNS]
    out["doi"] = out["doi"].map(normalize_doi)
    out.to_json(output_path, orient="records", lines=True)
    print(f"stored in {output_path}")


def schemamapping(columns_new, df, new_name):
    available_cols = [c for c in columns_new.keys() if c in df.columns]
    df2 = df[available_cols].copy()
    df2.rename(columns=columns_new, inplace=True)
    _write_schema_output(df2, new_name)


def _safe_nested(obj, path, default=None):
    """Safely get nested keys: handles None/NaN/non-dict/JSON-strings."""
    if obj is None or (isinstance(obj, float) and pd.isna(obj)):
        return default
    if isinstance(obj, str):
        try:
            obj = json.loads(obj)
        except Exception:
            return default
    if not isinstance(obj, Mapping):
        return default
    cur = obj
    for k in path:
        if not isinstance(cur, Mapping):
            return default
        cur = cur.get(k)
        if cur is None:
            return default
    return cur


def extract_author_names(authorships):
    if isinstance(authorships, str):
        try:
            authorships = json.loads(authorships)
        except Exception:
            return None
    if not isinstance(authorships, Sequence):
        return None
    names = []
    for a in authorships:
        if isinstance(a, Mapping):
            name = _safe_nested(a, ["author", "display_name"])
            if name:
                names.append(name)
    return names if names else None


def map_openalex(path: Path):
    df = pd.read_json(path, lines=True)
    if _already_target_shaped(df):
        _write_schema_output(df, "open_alex")
        return

    if "primary_location" in df.columns:
        df.loc[:, "journal"] = df["primary_location"].map(lambda x: _safe_nested(x, ["source", "display_name"]))
        df.loc[:, "publisher"] = df["primary_location"].map(lambda x: _safe_nested(x, ["display_name"]))
    if "biblio" in df.columns:
        df.loc[:, "volume"] = df["biblio"].map(lambda x: _safe_nested(x, ["volume"]))
        df.loc[:, "issue"] = df["biblio"].map(lambda x: _safe_nested(x, ["issue"]))
        df.loc[:, "first_page"] = df["biblio"].map(lambda x: _safe_nested(x, ["first_page"]))
        df.loc[:, "last_page"] = df["biblio"].map(lambda x: _safe_nested(x, ["last_page"]))
    if "authorships" in df.columns:
        df.loc[:, "authorships"] = df["authorships"].map(extract_author_names)
    if "keywords" in df.columns:
        def _keywords(items):
            if isinstance(items, list):
                return ", ".join(d.get("display_name", "") for d in items if isinstance(d, Mapping) and d.get("display_name"))
            return items
        df["keywords"] = df["keywords"].apply(_keywords)
    if "first_page" in df.columns:
        df["first_page"] = df["first_page"].astype("string").str.split(":").str[-1]
    if "last_page" in df.columns:
        df["last_page"] = df["last_page"].astype("string").str.split(":").str[-1]
        df["last_page"] = df["last_page"].fillna(df.get("first_page"))
    if "doi" in df.columns:
        df["doi"] = df["doi"].map(normalize_doi)

    columns_new = {
        "doi": "doi", "type": "type", "title": "title", "authorships": "authors",
        "publication_year": "publication_year", "journal": "journal", "publisher": "publisher",
        "keywords": "keywords", "volume": "volume", "issue": "issue", "first_page": "first_page",
        "last_page": "last_page", "referenced_works_count": "referenced_works_count",
        "cited_by_count": "cited_by_count",
    }
    schemamapping(columns_new, df, "open_alex")


def map_dblp(path: Path):
    df = pd.read_json(path, lines=True)
    if _already_target_shaped(df):
        _write_schema_output(df, "dblp")
        return

    columns_new = {
        "doi_list": "doi", "type": "type", "title": "title", "authors": "authors",
        "year": "publication_year", "journal": "journal", "publisher": "publisher",
        "keywords": "keywords", "volume": "volume", "issue": "issue", "first_page": "first_page",
        "last_page": "last_page", "referenced_works_count": "referenced_works_count",
        "cited_by_count": "cited_by_count",
    }
    df["keywords"] = np.nan
    df["referenced_works_count"] = np.nan
    df["cited_by_count"] = np.nan
    source_doi_col = "doi_list" if "doi_list" in df.columns else "doi"
    df["doi"] = df[source_doi_col].apply(lambda x: str(x[0]) if isinstance(x, list) and len(x) > 0 else x)
    schemamapping(columns_new, df, "dblp")


def map_crossref(path: Path):
    df = pd.read_json(path, lines=True)
    if _already_target_shaped(df):
        _write_schema_output(df, "crossref")
        return

    columns_new = {
        "doi": "doi", "type": "type", "title": "title", "authors": "authors",
        "publication_year": "publication_year", "journal": "journal", "publisher": "publisher",
        "abstract": "keywords", "volume": "volume", "issue": "issue", "first_page": "first_page",
        "last_page": "last_page", "reference-count": "referenced_works_count",
        "is-referenced-by-count": "cited_by_count",
    }
    if "page" in df.columns:
        df["first_page"] = df["page"].astype(str).str.split('-').str[0]
        df["last_page"] = df["page"].astype(str).str.split('-').str[-1]
    schemamapping(columns_new, df, "crossref")


map_openalex(OPENALEX_FILTERED_PATH)
map_dblp(DBLP_FILTERED_PATH)
map_crossref(CROSSREF_FILTERED_PATH)


stored in /Users/aaronsteiner/Documents/GitHub/PyDI/usecases/papers/output/schema_matching/open_alex.jsonl
stored in /Users/aaronsteiner/Documents/GitHub/PyDI/usecases/papers/output/schema_matching/dblp.jsonl
stored in /Users/aaronsteiner/Documents/GitHub/PyDI/usecases/papers/output/schema_matching/crossref.jsonl


In [6]:
# Exact dataset loading and author normalization from finding_a_matcher.ipynb cells 3-4,
# with paths adapted to the schema outputs produced above.
df_dblp = load_json(SCHEMA_OUTPUT_DIR / "dblp.jsonl", add_index=True, lines=True)
df_dblp.rename(columns={"dblp_id": "id"}, inplace=True)
df_crossref = load_json(SCHEMA_OUTPUT_DIR / "crossref.jsonl", add_index=True, lines=True)
df_crossref.rename(columns={"crossref_id": "id"}, inplace=True)
df_openalex = load_json(SCHEMA_OUTPUT_DIR / "open_alex.jsonl", add_index=True, lines=True)
df_openalex.rename(columns={"open_alex_id": "id"}, inplace=True)

df_dblp['authors_normalized'] = df_dblp['authors'].apply(normalize_authors_list)
df_crossref['authors_normalized'] = df_crossref['authors'].apply(normalize_authors_list)
df_openalex['authors_normalized'] = df_openalex['authors'].apply(normalize_authors_list)

DATASETS = {
    "dblp": df_dblp,
    "crossref": df_crossref,
    "open_alex": df_openalex,
}

for name, df in DATASETS.items():
    df.attrs["dataset_name"] = name
    print(f"{name}: {len(df):,} rows")
    display(df.head(2))

dblp: 60,591 rows


,id,doi,type,title,authors,publication_year,journal,publisher,keywords,volume,issue,first_page,last_page,referenced_works_count,cited_by_count,authors_normalized
0,dblp-00000,10.1145/3316781.3317924,inproceedings,Thread Weaving: Static Resource Scheduling for...,"[Hsuan Hsiao, Jason Helge Anderson]",2019,None,NaN,NaN,None,None,NaN,NaN,NaN,NaN,"[hsuan hsiao, jason helge anderson]"
1,dblp-00001,10.1109/dac.2018.8465925,inproceedings,LEAD: learning-enabled energy-aware dynamic vo...,"[Mark Clark, Avinash Kodi, Razvan C. Bunescu, ...",2018,None,NaN,NaN,None,None,1.0,6.0,NaN,NaN,"[mark clark, avinash kodi, razvan c. bunescu, ..."


crossref: 60,749 rows


,id,doi,type,title,authors,publication_year,journal,publisher,keywords,volume,issue,first_page,last_page,referenced_works_count,cited_by_count,authors_normalized
0,crossref-00000,10.5220/0006842100980105,inproceedings,A Simulation-driven Approach in Risk-aware Bus...,"[Ilaria Angela Amantea, Antonio Di Leva, Emi...",2018,None,SCITEPRESS - Science and Technology Publications,None,None,None,98,105,0,18,"[ilaria angela amantea, antonio di leva, emili..."
1,crossref-00001,10.1155/2018/6709607,article,Group Recommendation Systems Based on External...,"[Guang Fang, Lei Su, Di Jiang, Liping Wu]",2018,Wireless Communications and Mobile Computing,Wiley,With the development of social networks and on...,2018,1,None,None,35,14,"[guang fang, lei su, di jiang, liping wu]"


open_alex: 60,719 rows


,id,doi,type,title,authors,publication_year,journal,publisher,keywords,volume,issue,first_page,last_page,referenced_works_count,cited_by_count,authors_normalized
0,open_alex-00000,10.1093/nar/gky1131,article,STRING v11: protein–protein association networ...,"[Damian Szklarczyk, Annika L Gable, David Lyon...",2018,Nucleic Acids Research,NaN,"KEGG, Interaction network",47,D1,D607,D613,64,16038,"[damian szklarczyk, annika l gable, david lyon..."
1,open_alex-00001,10.1093/bioinformatics/bty191,article,Minimap2: pairwise alignment for nucleotide se...,[Heng Li],2018,Bioinformatics,NaN,Multiple sequence alignment,34,18,3094,3100,42,12292,[heng li]


In [7]:
profile_rows = []
for name, df in DATASETS.items():
    for col in TARGET_SCHEMA:
        profile_rows.append({
            "dataset": name,
            "column": col,
            "non_null": int(df[col].notna().sum()),
            "density": float(df[col].notna().mean()),
        })
profile = pd.DataFrame(profile_rows)
display(profile.pivot(index="column", columns="dataset", values="density").round(3))

dataset,crossref,dblp,open_alex
column,,,
authors,0.999,0.994,1.000
cited_by_count,1.000,0.000,1.000
doi,1.000,1.000,1.000
first_page,1.000,0.095,0.860
issue,0.488,0.459,0.484
journal,0.769,0.744,0.800
keywords,0.306,0.000,1.000
last_page,1.000,0.095,0.860
publication_year,1.000,1.000,1.000


## Part 2: Blocking

In [8]:
from PyDI.entitymatching.blocking import TokenBlocker
from PyDI.entitymatching.evaluation import EntityMatchingEvaluator
import ast
import re

IR_DIR = BASE_DIR / "IdentityResolution"
SPLIT_DIR = IR_DIR / "validation_sets" / "splits"
MATCH_DIR = IR_DIR / "matches"
BLOCKING_OUTPUT_DIR = EM_OUTPUT_DIR / "blocking" / "token_blocker_author_ngram2"
BLOCKING_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Exact winner from finding_a_blocker.ipynb:
# TokenBlocker(column='authors', ngram_size=2, ngram_type='word', preprocess=clean_authors)
# Set this to False after a first run if you want to reuse this notebook's cached materialized candidates.
REBUILD_BLOCKING_CANDIDATES = True


def load_split(pair_name: str, split: str) -> pd.DataFrame:
    path = SPLIT_DIR / pair_name / f"{pair_name}_{split}.csv"
    df = pd.read_csv(path)
    rename = {"id_dblp": "id1", "id_crossref": "id2", "id_openalex": "id2"}
    return df.rename(columns=rename)[["id1", "id2", "label"]]


def clean_authors(author_data):
    """Exact cleaner from finding_a_blocker.ipynb winner cells."""
    if not author_data:
        return ""

    name_list = []
    if isinstance(author_data, list):
        name_list = author_data
    elif isinstance(author_data, str):
        try:
            evaluated_data = ast.literal_eval(author_data)
            if isinstance(evaluated_data, list):
                name_list = evaluated_data
            else:
                name_list = [str(evaluated_data)]
        except (ValueError, SyntaxError):
            name_list = author_data.split(',')
    else:
        name_list = [str(author_data)]

    clean_names = [str(name).strip() for name in name_list if name and str(name).strip()]
    text = " ".join(clean_names)
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    return " ".join(text.split())


def materialize_winner_blocker(pair_name: str, df_left: pd.DataFrame, df_right: pd.DataFrame) -> pd.DataFrame:
    cache_path = BLOCKING_OUTPUT_DIR / f"{pair_name}_author_token_blocking_candidates.csv"
    if cache_path.exists() and not REBUILD_BLOCKING_CANDIDATES:
        print(f"{pair_name}: loading cached candidates from {cache_path}")
        return pd.read_csv(cache_path)

    blocker_token = TokenBlocker(
        df_left,
        df_right,
        column='authors',
        output_dir=BLOCKING_OUTPUT_DIR / pair_name,
        id_column='id',
        ngram_size=2,
        ngram_type='word',
        preprocess=clean_authors,
    )
    candidates = blocker_token.materialize()
    candidates.to_csv(cache_path, index=False)
    print(f"{pair_name}: materialized {len(candidates):,} candidates with exact winner blocker")
    return candidates


candidates_crossref = materialize_winner_blocker("dblp_crossref", df_dblp, df_crossref)
candidates_openalex = materialize_winner_blocker("dblp_openalex", df_dblp, df_openalex)

df_validation_crossref = load_split("dblp_crossref", "val")
df_validation_openalex = load_split("dblp_openalex", "val")
df_test_crossref = load_split("dblp_crossref", "test")
df_test_openalex = load_split("dblp_openalex", "test")

# Backward-compatible names used by later summary cells.
test_crossref = df_test_crossref
test_openalex = df_test_openalex

display(candidates_crossref.head())

/Users/aaronsteiner/Documents/GitHub/PyDI/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[INFO ] PyDI.entitymatching.blocking.token_blocking.TokenBlocker - created 323944 token keys for first dataset
[INFO ] PyDI.entitymatching.blocking.token_blocking.TokenBlocker - created 326502 token keys for second dataset
[INFO ] PyDI.entitymatching.blocking.token_blocking.TokenBlocker - created 255631 blocks from token keys
[INFO ] PyDI.entitymatching.blocking.token_blocking.TokenBlocker - Debug results written to file: /Users/aaronsteiner/Documents/GitHub/PyDI/usecases/papers/output/entity_matching/blocking/token_blocker_author_ngram2/dblp_crossref/debugResultsBlocking_TokenBlocker.csv


dblp_crossref: materialized 855,308 candidates with exact winner blocker


[INFO ] PyDI.entitymatching.blocking.token_blocking.TokenBlocker - created 323944 token keys for first dataset
[INFO ] PyDI.entitymatching.blocking.token_blocking.TokenBlocker - created 313964 token keys for second dataset
[INFO ] PyDI.entitymatching.blocking.token_blocking.TokenBlocker - created 260527 blocks from token keys
[INFO ] PyDI.entitymatching.blocking.token_blocking.TokenBlocker - Debug results written to file: /Users/aaronsteiner/Documents/GitHub/PyDI/usecases/papers/output/entity_matching/blocking/token_blocker_author_ngram2/dblp_openalex/debugResultsBlocking_TokenBlocker.csv


dblp_openalex: materialized 905,386 candidates with exact winner blocker


FileNotFoundError: [Errno 2] No such file or directory: '/Users/aaronsteiner/Documents/GitHub/PyDI/usecases/papers/IdentityResolution/validation_sets/splits/dblp_crossref/dblp_crossref_val.csv'

In [ ]:
blocking_summary = []
for pair_name, candidates, right_df, test_pairs in [
    ("dblp_crossref", candidates_crossref, df_crossref, test_crossref),
    ("dblp_openalex", candidates_openalex, df_openalex, test_openalex),
]:
    metrics = EntityMatchingEvaluator.evaluate_blocking(
        candidate_pairs=candidates[["id1", "id2"]],
        test_pairs=test_pairs,
        total_possible_pairs=len(df_dblp) * len(right_df),
        out_dir=EM_OUTPUT_DIR / "blocking" / pair_name,
    )
    blocking_summary.append({"pair": pair_name, **metrics})

blocking_summary = pd.DataFrame(blocking_summary)
display(blocking_summary[["pair", "total_candidates", "pair_completeness", "pair_quality", "reduction_ratio"]])

[INFO ] root -   Pair Completeness: 0.975
[INFO ] root -   Pair Quality:      0.006
[INFO ] root -   Reduction Ratio:   0.999768
[INFO ] root -   True Matches Found: 4875/5000
[INFO ] root - Blocking evaluation complete!
[INFO ] root -   Pair Completeness: 0.985
[INFO ] root -   Pair Quality:      0.005
[INFO ] root -   Reduction Ratio:   0.999754
[INFO ] root -   True Matches Found: 4924/5000
[INFO ] root - Blocking evaluation complete!


,pair,total_candidates,pair_completeness,pair_quality,reduction_ratio
0,dblp_crossref,855308,0.9750,0.005700,0.999768
1,dblp_openalex,905386,0.9848,0.005439,0.999754


## Part 3: Entity Matching

In [ ]:
from PyDI.entitymatching.comparators import StringComparator, DateComparator
from PyDI.entitymatching import FeatureExtractor, MLBasedMatcher

extractor_refined = FeatureExtractor([
    StringComparator("title", "overlap", tokenization="word", preprocess=str.lower),
    StringComparator("title", "jaro_winkler", preprocess=str.lower),
    StringComparator("title", "ratcliff_obershelp", preprocess=str.lower),
    StringComparator("title", "mra", preprocess=str.lower),
    StringComparator("title", "prefix", preprocess=str.lower),
    StringComparator("title", "jaccard", tokenization="ngram_3", preprocess=str.lower),
    StringComparator(
        column="authors_normalized",
        similarity_function="jaccard",
        tokenization="word",
        list_strategy="concatenate",
    ),
])

feature_names = [
    "title_overlap_word",
    "title_jaro_winkler",
    "title_ratcliff_obershelp",
    "title_mra_phonetic",
    "title_prefix",
    "title_jaccard_ngram3",
    "authors_bow_jaccard_word",
]

print("Refined feature set used by the final XGBoost model:")
display(pd.DataFrame({"feature": feature_names}))

Refined feature set used by the final XGBoost model:


,feature
0,title_overlap_word
1,title_jaro_winkler
2,title_ratcliff_obershelp
3,title_mra_phonetic
4,title_prefix
5,title_jaccard_ngram3
6,authors_bow_jaccard_word


In [ ]:
RUN_FULL_XGBOOST_INFERENCE = False

if RUN_FULL_XGBOOST_INFERENCE:
    from xgboost import XGBClassifier

    train_crossref = pd.read_csv(SPLIT_DIR / "negative_mining" / "dblp_crossref_enriched_dblp_crossref_train_enriched.csv")
    train_openalex = pd.read_csv(SPLIT_DIR / "negative_mining" / "dblp_openalex_enriched_dblp_openalex_train_enriched.csv")

    features_crossref = extractor_refined.create_features(df_dblp, df_crossref, train_crossref, "id", train_crossref["label"])
    features_openalex = extractor_refined.create_features(df_dblp, df_openalex, train_openalex, "id", train_openalex["label"])

    xgb_crossref = XGBClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    xgb_openalex = XGBClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    xgb_crossref.fit(features_crossref.drop(["label", "id1", "id2"], axis=1), features_crossref["label"])
    xgb_openalex.fit(features_openalex.drop(["label", "id1", "id2"], axis=1), features_openalex["label"])

    matcher_crossref = MLBasedMatcher(extractor_refined)
    matcher_openalex = MLBasedMatcher(extractor_refined)
    matches_crossref = matcher_crossref.match(df_dblp, df_crossref, candidates_crossref, "id", xgb_crossref, threshold=0.5, use_probabilities=True)
    matches_openalex = matcher_openalex.match(df_dblp, df_openalex, candidates_openalex, "id", xgb_openalex, threshold=0.5, use_probabilities=True)
else:
    matches_crossref = pd.read_csv(MATCH_DIR / "matches_dblp-crossref.csv")
    matches_openalex = pd.read_csv(MATCH_DIR / "matches_dblp-openalex.csv")
    matches_crossref = matches_crossref[["id1", "id2", "score", "notes"]].copy()
    matches_openalex = matches_openalex[["id1", "id2", "score", "notes"]].copy()

print(f"DBLP-Crossref matches: {len(matches_crossref):,}")
print(f"DBLP-OpenAlex matches: {len(matches_openalex):,}")
display(matches_crossref.head())

DBLP-Crossref matches: 48,889
DBLP-OpenAlex matches: 54,063


,id1,id2,score,notes
0,dblp-14817,crossref-41952,0.999976,ml_classifier=XGBClassifier
1,dblp-29108,crossref-17704,0.999976,ml_classifier=XGBClassifier
2,dblp-14797,crossref-41811,0.999976,ml_classifier=XGBClassifier
3,dblp-24152,crossref-58835,0.999976,ml_classifier=XGBClassifier
4,dblp-40211,crossref-28332,0.999976,ml_classifier=XGBClassifier


In [ ]:
# Exact evaluation pattern from finding_a_matcher.ipynb cells 63-65.
df_val_enriched_crossref = pd.read_csv(SPLIT_DIR / "negative_mining" / "dblp_crossref_enriched_dblp_crossref_val_enriched.csv")
df_val_enriched_openalex = pd.read_csv(SPLIT_DIR / "negative_mining" / "dblp_openalex_enriched_dblp_openalex_val_enriched.csv")

xgb_eval_rows = []
for pair_name, matches, original_val, enriched_val in [
    ("dblp_crossref", matches_crossref, df_validation_crossref, df_val_enriched_crossref),
    ("dblp_openalex", matches_openalex, df_validation_openalex, df_val_enriched_openalex),
]:
    original_metrics = EntityMatchingEvaluator.evaluate_matching(
        correspondences=matches,
        test_pairs=original_val,
    )
    enriched_metrics = EntityMatchingEvaluator.evaluate_matching(
        correspondences=matches,
        test_pairs=enriched_val,
    )
    xgb_eval_rows.append({"pair": pair_name, "validation_set": "original", **original_metrics})
    xgb_eval_rows.append({"pair": pair_name, "validation_set": "hard_negative_enriched", **enriched_metrics})

matching_summary = pd.DataFrame(xgb_eval_rows)
display(matching_summary[["pair", "validation_set", "precision", "recall", "f1", "accuracy", "filtered_correspondences"]])

[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  4834
[INFO ] root -   True Negatives:  5000
[INFO ] root -   False Positives: 0
[INFO ] root -   False Negatives: 166
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.983
[INFO ] root -   Precision: 1.000
[INFO ] root -   Recall:    0.967
[INFO ] root -   F1-Score:  0.983
[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  4901
[INFO ] root -   True Negatives:  10424
[INFO ] root -   False Positives: 19
[INFO ] root -   False Negatives: 170
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.988
[INFO ] root -   Precision: 0.996
[INFO ] root -   Recall:    0.966
[INFO ] root -   F1-Score:  0.981
[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  4834
[INFO ] root -   True Negatives:  5000
[INFO ] root -   False Positives: 0
[INFO ] root -   False Negatives: 166
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.983
[INFO ] root -   Precision: 1

,pair,validation_set,precision,recall,f1,accuracy,filtered_correspondences
0,dblp_crossref,original,1.000000,0.966800,0.983120,0.983400,48889
1,dblp_crossref,hard_negative_enriched,0.996138,0.966476,0.981083,0.987817,48889
2,dblp_openalex,original,1.000000,0.966800,0.983120,0.983400,54063
3,dblp_openalex,hard_negative_enriched,0.995917,0.966132,0.980798,0.987225,54063


In [ ]:
from PyDI.entitymatching import GreedyOneToOneMatchingAlgorithm

# Exact final declustering from finding_a_matcher.ipynb cells 74/76.
greedy_matching_crossref = GreedyOneToOneMatchingAlgorithm()
greedy_matches_crossref = greedy_matching_crossref.cluster(matches_crossref)

greedy_matching_openalex = GreedyOneToOneMatchingAlgorithm()
greedy_matches_openalex = greedy_matching_openalex.cluster(matches_openalex)

# Exact final test files from finding_a_matcher.ipynb cells 79/80, with local paths.
df_test_enriched_crossref = pd.read_csv(SPLIT_DIR / "negative_mining" / "dblp_crossref_enriched_dblp_crossref_test_enriched.csv")
df_test_enriched_openalex = pd.read_csv(SPLIT_DIR / "negative_mining" / "dblp_openalex_enriched_dblp_openalex_test_enriched.csv")

final_eval_crossref = EntityMatchingEvaluator.evaluate_matching(
    correspondences=greedy_matches_crossref,
    test_pairs=df_test_enriched_crossref,
)
final_eval_openalex = EntityMatchingEvaluator.evaluate_matching(
    correspondences=greedy_matches_openalex,
    test_pairs=df_test_enriched_openalex,
)

final_matching_summary = pd.DataFrame([
    {"pair": "dblp_crossref", **final_eval_crossref},
    {"pair": "dblp_openalex", **final_eval_openalex},
])
display(final_matching_summary[["pair", "precision", "recall", "f1", "accuracy", "filtered_correspondences"]])

# These are the exact final correspondence variables used downstream.
correspondences_crossref_final = greedy_matches_crossref
correspondences_openalex_final = greedy_matches_openalex

[INFO ] root - Filtered correspondences: 48889 -> 48889 (threshold=0.0)
[INFO ] root - Greedy matching: 48889 -> 48889 correspondences (97778 entities matched)
[INFO ] root - GreedyOneToOneMatchingAlgorithm: 48889 -> 48889 correspondences
[INFO ] root - GreedyOneToOneMatchingAlgorithm: 97778 -> 97778 entities
[INFO ] root - Filtered correspondences: 54063 -> 54063 (threshold=0.0)
[INFO ] root - Greedy matching: 54063 -> 54063 correspondences (108126 entities matched)
[INFO ] root - GreedyOneToOneMatchingAlgorithm: 54063 -> 54063 correspondences
[INFO ] root - GreedyOneToOneMatchingAlgorithm: 108126 -> 108126 entities
[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  4913
[INFO ] root -   True Negatives:  10535
[INFO ] root -   False Positives: 24
[INFO ] root -   False Negatives: 181
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.987
[INFO ] root -   Precision: 0.995
[INFO ] root -   Recall:    0.964
[INFO ] root -   F1-Score:  0.980
[INFO ] root - C

,pair,precision,recall,f1,accuracy,filtered_correspondences
0,dblp_crossref,0.995139,0.964468,0.979563,0.986903,48889
1,dblp_openalex,0.996141,0.971287,0.983557,0.989099,54063


## Part 4: Data Fusion

In [ ]:
from PyDI.fusion import (
    DataFusionStrategy,
    DataFusionEngine,
    DataFusionEvaluator,
    longest_string,
    prefer_higher_trust,
    union,
    tokenized_match,
    year_only_match,
    most_recent,
    voting,
    maximum,
    intersection,
    weighted_voting,
    shortest_string,
)

# Recreate the shape produced by PyDI load_json(..., add_index=True, name=name)
# in fusion/main.ipynb: source-specific ID columns and no generic entity-matching id.
def fusion_input_from_mapped(df: pd.DataFrame, name: str) -> pd.DataFrame:
    out = df.drop(columns=["id", "authors_normalized"], errors="ignore").copy()
    out.insert(0, f"{name}_id", df["id"].astype(str).to_numpy())
    out = out.drop(columns=["publisher", "cited_by_count"], errors="ignore")
    out.attrs["dataset_name"] = name
    return out

loaded = {
    "dblp": fusion_input_from_mapped(df_dblp, "dblp"),
    "crossref": fusion_input_from_mapped(df_crossref, "crossref"),
    "open_alex": fusion_input_from_mapped(df_openalex, "open_alex"),
}

match_files = {
    "dblp-crossref": correspondences_crossref_final,
    "dblp-openalex": correspondences_openalex_final,
}

filtered_loaded = loaded

df_test_set = pd.read_json(BASE_DIR / "fusion" / "fusion_test.jsonl", lines=True)

# Exact gold-standard dtype normalization from fusion/main.ipynb cell 3.
for col in ["issue", "first_page", "last_page", "volume"]:
    if col in df_test_set.columns:
        ser = df_test_set[col]
        ser_num = pd.to_numeric(ser, errors="coerce")
        ser_int = ser_num.astype("Int64")
        ser_str = ser_int.astype("string")
        mask = ser_num.isna() & ser.notna()
        ser_str.loc[mask] = ser.loc[mask].astype(str)
        df_test_set[col] = ser_str

all_correspondences = pd.concat(
    [
        match_files["dblp-crossref"][["id1", "id2", "score"]],
        match_files["dblp-openalex"][["id1", "id2", "score"]],
    ],
    ignore_index=True,
)
print(f"Correspondences passed to fusion: {len(all_correspondences):,}")

Correspondences passed to fusion: 102,952


In [ ]:
import ast
import re
from PyDI.fusion import FusionResult

# Exact helper functions and V5 setup from fusion/main.ipynb cells 20, 27, 32, and 37.
def _normalize_pages(df):
    df = df.copy()
    for col in ["first_page", "last_page"]:
        if col not in df.columns:
            continue
        def _fix(val):
            if isinstance(val, str):
                s = val.strip()
                if re.fullmatch(r"\d+:\d+", s):
                    a, b = s.split(":")
                    nums = sorted([int(a), int(b)])
                    return str(nums[0]) if col == "first_page" else str(nums[1])
                if s.isdigit():
                    return s
            return val
        df[col] = df[col].apply(_fix)
    for col in ["first_page", "last_page"]:
        if col in df.columns:
            ser_num = pd.to_numeric(df[col], errors="coerce")
            ser_int = ser_num.astype("Int64")
            df[col] = ser_int.astype("string")
    return df


def normalize_authors_field(df):
    df = df.copy()
    if 'authors' not in df.columns:
        return df
    def _norm(val):
        if isinstance(val, list):
            parts = [str(v).strip() for v in val if str(v).strip()]
        elif isinstance(val, str):
            s = val.strip()
            try:
                if s.startswith('[') and s.endswith(']'):
                    parsed = ast.literal_eval(s)
                    if isinstance(parsed, list):
                        parts = [str(v).strip() for v in parsed if str(v).strip()]
                    else:
                        parts = []
                else:
                    parts = [p.strip() for p in s.split(',') if p.strip()]
            except Exception:
                parts = [p.strip() for p in s.split(',') if p.strip()]
        else:
            parts = []
        parts = [' '.join(p.split()) for p in parts if p]
        return parts
    df['authors'] = df['authors'].apply(_norm)
    return df


def normalize_journal_field(df):
    df = df.copy()
    if 'journal' not in df.columns:
        return df
    def _norm(val):
        if not isinstance(val, str):
            return val
        s = val.strip()
        if s.lower().startswith('the '):
            s = s[4:]
        s = s.replace('&amp;', '&')
        return s
    df['journal'] = df['journal'].apply(_norm)
    return df


def normalize_reference_counts(df):
    df = df.copy()
    if 'referenced_works_count' in df.columns:
        ser = pd.to_numeric(df['referenced_works_count'], errors='coerce')
        ser = ser.where(ser != 0, pd.NA)
        df['referenced_works_count'] = ser
    return df


def normalize_title_field(df):
    df = df.copy()
    if 'title' not in df.columns:
        return df
    def _norm(val):
        if not isinstance(val, str):
            return val
        return val.rstrip(' .')
    df['title'] = df['title'].apply(_norm)
    return df


keyword_trust = {'crossref': 1, 'open_alex': 3, 'dblp': 1}
normalized_loaded_v5 = {}
for name, df in filtered_loaded.items():
    trust_val = df.attrs.get('trust')
    norm_df = normalize_reference_counts(
        normalize_journal_field(normalize_title_field(normalize_authors_field(_normalize_pages(df))))
    )
    if trust_val is not None:
        norm_df.attrs['trust'] = trust_val
    normalized_loaded_v5[name] = norm_df

for name, trust_val in [('dblp', 1), ('crossref', 3), ('open_alex', 1)]:
    if name in normalized_loaded_v5:
        normalized_loaded_v5[name].attrs['trust'] = trust_val

gold_norm_v5 = normalize_reference_counts(
    normalize_journal_field(normalize_title_field(normalize_authors_field(_normalize_pages(df_test_set))))
)

strategy_v5 = DataFusionStrategy("Version_5_JournalClean")
strategy_v5.add_attribute_fuser("type", voting)
strategy_v5.add_attribute_fuser("title", longest_string)
strategy_v5.add_attribute_fuser("authors", prefer_higher_trust)
strategy_v5.add_attribute_fuser("publication_year", most_recent)
strategy_v5.add_attribute_fuser("journal", voting)
strategy_v5.add_attribute_fuser("keywords", prefer_higher_trust, trust_map={'crossref': 1, 'open_alex': 3, 'dblp': 1})
strategy_v5.add_attribute_fuser("volume", voting)
strategy_v5.add_attribute_fuser("issue", voting)
strategy_v5.add_attribute_fuser("first_page", voting)
strategy_v5.add_attribute_fuser("last_page", prefer_higher_trust, trust_map={'crossref': 3, 'open_alex': 1, 'dblp': 1})
strategy_v5.add_attribute_fuser("referenced_works_count", prefer_higher_trust, trust_map={'crossref': 3, 'open_alex': 1, 'dblp': 1})

engine_v5 = DataFusionEngine(strategy_v5, debug=True, debug_file=FUSION_OUTPUT_DIR / "fusion_debug_v5.jsonl")
fused_v5 = engine_v5.run(
    datasets=list(normalized_loaded_v5.values()),
    correspondences=[
        match_files['dblp-crossref'][['id1', 'id2', 'score']],
        match_files['dblp-openalex'][['id1', 'id2', 'score']],
    ],
    id_column={'dblp': 'dblp_id', 'crossref': 'crossref_id', 'open_alex': 'open_alex_id'},
)

fused_v5["_id"] = fused_v5["_fusion_sources"].apply(lambda sources: "+".join(sorted(sources)))
fused_v5.to_json(FUSION_OUTPUT_DIR / "papers_fused_v5.jsonl", orient="records", lines=True)
print(f"Fused records: {len(fused_v5):,}")
print("Saved to", FUSION_OUTPUT_DIR / "papers_fused_v5.jsonl")
display(fused_v5.head())

[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'type' using rule 'voting'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'title' using rule 'longest_string'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'authors' using rule 'prefer_higher_trust'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'publication_year' using rule 'most_recent'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'journal' using rule 'voting'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'keywords' using rule 'prefer_higher_trust'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'volume' using rule 'voting'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'issue' using rule 'voting'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'first_page' using rule 'voting'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'last_page' using rule 'prefer_higher_trust'
[INFO ] PyDI.fusion.s

Fused records: 54,717
Saved to /Users/aaronsteiner/Documents/GitHub/PyDI/usecases/papers/output/data_fusion/papers_fused_v5.jsonl


,_id,_fusion_sources,_fusion_source_datasets,authors,crossref_id,dblp_id,doi,first_page,issue,journal,keywords,last_page,open_alex_id,publication_year,referenced_works_count,title,type,volume,_fusion_confidence,_fusion_metadata
0,crossref-28830+dblp-00000+open_alex-42427,"[dblp-00000, crossref-28830, open_alex-42427]","[dblp, crossref, open_alex]","[Hsuan Hsiao, Jason Anderson]",crossref-28830,dblp-00000,10.1145/3316781.3317924,1,None,None,Weaving,6,open_alex-42427,2019,11.0,Thread Weaving: Static Resource Scheduling for...,inproceedings,None,0.605350,"{'_id_rule': 'first_non_null', '_id_inputs': [..."
1,crossref-09148+dblp-00001+open_alex-18938,"[dblp-00001, crossref-09148, open_alex-18938]","[dblp, crossref, open_alex]","[Mark Clark, Avinash Kodi, Razvan Bunescu, Ahm...",crossref-09148,dblp-00001,10.1145/3195970.3196068,1,None,None,Frequency scaling,6,open_alex-18938,2018,23.0,LEAD: learning-enabled energy-aware dynamic vo...,inproceedings,None,0.609380,"{'_id_rule': 'first_non_null', '_id_inputs': [..."
2,crossref-09142+dblp-00002+open_alex-17364,"[dblp-00002, crossref-09142, open_alex-17364]","[dblp, crossref, open_alex]","[Yu-Kai Chuang, Kuan-Jung Chen, Kun-Lin Lin, S...",crossref-09142,dblp-00002,10.1145/3195970.3196093,1,None,None,Network routing,6,open_alex-17364,2018,13.0,PlanarONoC: concurrent placement and routing c...,inproceedings,None,0.607996,"{'_id_rule': 'first_non_null', '_id_inputs': [..."
3,crossref-28774+dblp-00003+open_alex-33390,"[dblp-00003, crossref-28774, open_alex-33390]","[dblp, crossref, open_alex]","[Duy-Thanh Nguyen, Nhut-Minh Ho, Ik-Joon Chang]",crossref-28774,dblp-00003,10.1145/3316781.3317915,1,None,None,"Dram, Parity bit",6,open_alex-33390,2019,28.0,St-DRC: Stretchable DRAM Refresh Controller wi...,inproceedings,None,0.609402,"{'_id_rule': 'first_non_null', '_id_inputs': [..."
4,crossref-09125+dblp-00004+open_alex-13266,"[dblp-00004, crossref-09125, open_alex-13266]","[dblp, crossref, open_alex]","[Kosuke Watanabe, Eunsuk Kang, Chung-Wei Lin, ...",crossref-09125,dblp-00004,10.1145/3195970.3199856,1,None,None,"Runtime Verification, Safety Assurance",6,open_alex-13266,2018,19.0,Runtime monitoring for safety of intelligent v...,inproceedings,None,0.577778,"{'_id_rule': 'first_non_null', '_id_inputs': [..."


In [ ]:
# Exact evaluation alignment from fusion/main.ipynb cells 13/14/39, using V5 sources.
fusion_sources = fused_v5[["doi", "_fusion_sources"]].dropna(subset=["doi"])
augmented_gold = df_test_set.copy().merge(fusion_sources, on="doi", how="left")

augmented_gold["df_test_set_id"] = augmented_gold["_fusion_sources"].apply(
    lambda sources: "+".join(sorted(sources)) if isinstance(sources, list) else None
)

evaluator = DataFusionEvaluator(strategy_v5, debug=True, fusion_debug_logs=FUSION_OUTPUT_DIR / "fusion_debug_v5.jsonl")
baseline_metrics = evaluator.evaluate(
    fused_df=fused_v5,
    fused_id_column="_id",
    gold_df=augmented_gold,
    gold_id_column="df_test_set_id",
)

print("Fusion V5 metrics")
for key, value in baseline_metrics.items():
    print(f"{key}: {value}")

[INFO ] PyDI.fusion.evaluation - Fusion evaluation debug logging enabled; refer to fusion_evaluation_debug.jsonl for mismatch details.
[INFO ] PyDI.fusion.evaluation - Starting fusion evaluation
[INFO ] PyDI.fusion.evaluation - Evaluation complete: 0.789 overall accuracy (881/1116)
[INFO ] PyDI.fusion.evaluation - Evaluation mismatches by attribute (debug): 235 total
[INFO ] PyDI.fusion.evaluation - 	Attribute                        |  Errors | Percentage
[INFO ] PyDI.fusion.evaluation - 	───────────────────────────────────────────────────────
[INFO ] PyDI.fusion.evaluation - 	keywords                         |      66 |     28.09%%
[INFO ] PyDI.fusion.evaluation - 	title                            |      37 |     15.74%%
[INFO ] PyDI.fusion.evaluation - 	referenced_works_count           |      30 |     12.77%%
[INFO ] PyDI.fusion.evaluation - 	type                             |      27 |     11.49%%
[INFO ] PyDI.fusion.evaluation - 	journal                          |      23 |      9.

Fusion V5 metrics
overall_accuracy: 0.7894265232974911
macro_accuracy: 0.7722882171234392
num_evaluated_records: 100
num_evaluated_attributes: 12
total_evaluations: 1116
total_correct: 881
referenced_works_count_accuracy: 0.6739130434782609
referenced_works_count_count: 92
last_page_accuracy: 0.8617021276595744
last_page_count: 94
journal_accuracy: 0.7676767676767676
journal_count: 99
first_page_accuracy: 0.8541666666666666
first_page_count: 96
volume_accuracy: 1.0
volume_count: 91
keywords_accuracy: 0.0
keywords_count: 66
title_accuracy: 0.63
title_count: 100
type_accuracy: 0.73
type_count: 100
publication_year_accuracy: 0.93
publication_year_count: 100
authors_accuracy: 0.82
authors_count: 100
issue_accuracy: 1.0
issue_count: 78
doi_accuracy: 1.0
doi_count: 100


## Outputs

The consolidated workflow writes schema-mapped intermediate files to `output/schema_matching`, entity-matching diagnostics to `output/entity_matching`, and the fused JSONL dataset plus debug logs to `output/data_fusion`.